In [1]:
import openpyxl
from scipy import stats
import numpy as np

In [2]:
def load_improvements(path):
    """Returns dict: {dataset_name: best_improve%} from the 'Summary' sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["Summary"]
    rows = list(ws.iter_rows(values_only=True))
    header = rows[0]
    idx_name = header.index("Dataset")
    idx_imp  = header.index("Best Improve%")
    return {r[idx_name]: r[idx_imp] for r in rows[1:] if r[idx_name] is not None}

In [3]:

v1_path = "sa_v1_results.xlsx"
v2_path = "sa_v2_results.xlsx"
v3_path = "sa_v3_results.xlsx"

In [4]:
v1_imp = load_improvements(v1_path)
v2_imp = load_improvements(v2_path)
v3_imp = load_improvements(v3_path)

In [5]:
def compare(name_a, dict_a, name_b, dict_b):
    common = sorted(set(dict_a) & set(dict_b))
    a = np.array([dict_a[k] for k in common])
    b = np.array([dict_b[k] for k in common])
    diff = b - a

    print(f"\n{'='*70}")
    print(f"{name_a} vs {name_b}   (paired datasets: {len(common)})")
    print(f"{'='*70}")
    print(f"{name_a} mean improve%: {a.mean():.2f}  |  {name_b} mean improve%: {b.mean():.2f}")
    print(f"Mean difference ({name_b} - {name_a}): {diff.mean():.2f}  (std: {diff.std():.2f})")

    t_stat, t_p = stats.ttest_rel(b, a)
    print(f"\nPaired t-test:      t={t_stat:.3f}, p={t_p:.5f}")

    w_stat, w_p = stats.wilcoxon(b, a)
    print(f"Wilcoxon test:      W={w_stat:.3f}, p={w_p:.5f}")

    alpha = 0.05
    print(f"\nAt alpha={alpha}:")
    print(f"  t-test:    {'SIGNIFICANT' if t_p<alpha else 'not significant'}")
    print(f"  Wilcoxon:  {'SIGNIFICANT' if w_p<alpha else 'not significant'}")
    return {"pair": f"{name_a} vs {name_b}", "mean_diff": diff.mean(),
            "t_p": t_p, "w_p": w_p}

results_summary = []
results_summary.append(compare("V1", v1_imp, "V2", v2_imp))
results_summary.append(compare("V1", v1_imp, "V3", v3_imp))
results_summary.append(compare("V2", v2_imp, "V3", v3_imp))



V1 vs V2   (paired datasets: 45)
V1 mean improve%: 74.98  |  V2 mean improve%: 72.88
Mean difference (V2 - V1): -2.10  (std: 7.66)

Paired t-test:      t=-1.814, p=0.07642
Wilcoxon test:      W=399.000, p=0.18093

At alpha=0.05:
  t-test:    not significant
  Wilcoxon:  not significant

V1 vs V3   (paired datasets: 45)
V1 mean improve%: 74.98  |  V3 mean improve%: 67.05
Mean difference (V3 - V1): -7.93  (std: 8.28)

Paired t-test:      t=-6.352, p=0.00000
Wilcoxon test:      W=78.000, p=0.00000

At alpha=0.05:
  t-test:    SIGNIFICANT
  Wilcoxon:  SIGNIFICANT

V2 vs V3   (paired datasets: 45)
V2 mean improve%: 72.88  |  V3 mean improve%: 67.05
Mean difference (V3 - V2): -5.84  (std: 4.29)

Paired t-test:      t=-9.015, p=0.00000
Wilcoxon test:      W=0.000, p=0.00000

At alpha=0.05:
  t-test:    SIGNIFICANT
  Wilcoxon:  SIGNIFICANT


In [6]:
# ── Compact summary table at the end ──────────────────────────────
print(f"\n\n{'='*70}")
print("SUMMARY TABLE")
print(f"{'='*70}")
print(f"{'Comparison':<15} {'Mean Diff':>12} {'t-test p':>12} {'Wilcoxon p':>12}")
for r in results_summary:
    print(f"{r['pair']:<15} {r['mean_diff']:>12.2f} {r['t_p']:>12.5f} {r['w_p']:>12.5f}")



SUMMARY TABLE
Comparison         Mean Diff     t-test p   Wilcoxon p
V1 vs V2               -2.10      0.07642      0.18093
V1 vs V3               -7.93      0.00000      0.00000
V2 vs V3               -5.84      0.00000      0.00000
